# CropGuard: training notebook (free Colab GPU)

**Goal:** train a plant-disease classifier you can explain to examiners, evaluate it honestly (lab vs field), calibrate its confidence, and export it to ONNX for the web app.

**Runs in two places (same notebook):**
- **Google Colab (easiest, free GPU):** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`. You will be asked once to upload your `kaggle.json`.
- **Your own PC in VS Code:** open this file in VS Code, select your `.venv` kernel, run all cells. Needs an NVIDIA GPU for a sensible training time (see `docs/SETUP_AND_TRAINING_GUIDE.md`). The result is installed into `ml/models/cropguard/` automatically.

Kaggle token: Kaggle -> Settings -> API -> Create New Token (downloads `kaggle.json`).

| Step | What it does | Why a professor will ask |
|---|---|---|
| 1 | Download PlantVillage (15 classes) | Which data? |
| 2 | **Leaf-level** train/val/test split + near-duplicate leak check | Is your test score inflated? |
| 3 | Augmentation incl. **background replacement** | Why would it work outside the lab? |
| 4 | EfficientNet-B0 transfer learning (two phases) + baselines | Why this model? Compared to what? |
| 5 | Metrics: accuracy, macro-F1, top-3, confusion matrix | How accurate, per class? |
| 6 | **Calibration** (ECE + temperature scaling) | Can I trust "94% confident"? |
| 7 | **Field test** on PlantDoc, then field fine-tuning | Does it work on real photos? |
| 8 | Risk-coverage curve to pick the confidence threshold | Why 60%? |
| 9 | Grad-CAM | Is it looking at the lesion? |
| 10 | ONNX export + verification | How is it deployed? |

Set `QUICK = True` in the config cell for a 5-minute smoke test on a small subset (numbers will be meaningless; use it only to check that everything runs).

In [ ]:
!nvidia-smi -L
!pip -q install timm onnx onnxruntime grad-cam scikit-learn matplotlib pandas kaggle

In [ ]:
import os, sys, json, random, re, glob, time, math, shutil, zipfile, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

QUICK = False          # True = tiny smoke test
RUN_BASELINES = True   # simple CNN + ResNet50 comparison (adds ~20 min)
MY_PHOTOS_DIR = None   # optional: folder of YOUR field photos for training, sub-folders named like the classes (e.g. 'my_photos')
MY_TEST_DIR = None     # optional: folder of YOUR photos used ONLY for testing (different leaves from MY_PHOTOS_DIR)
PRETRAINED_DIR = None  # optional: community model folder to compare against, e.g. '../ml/models/community'
CFG = dict(img=224, bs=64, workers=(0 if os.name == 'nt' else 2), epochs_head=3, epochs_ft=10, lr_head=1e-3, lr_ft=2e-4,
           base_epochs_cnn=8, base_epochs_r50_head=1, base_epochs_r50_ft=3)
SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', dev, '| torch', torch.__version__)
IN_COLAB = 'google.colab' in sys.modules
WORK = Path('/content') if IN_COLAB else Path('work'); WORK.mkdir(exist_ok=True)
OUT = WORK / 'cropguard_artifacts'; OUT.mkdir(exist_ok=True)
if dev.type == 'cpu': print('WARNING: no GPU found. Full training on a CPU takes many hours. Use Colab, or set QUICK = True to just test the pipeline.')

## 1. Data: PlantVillage (emmarex/plantdisease)

In [ ]:
KAGGLE_JSON = Path.home() / '.kaggle' / 'kaggle.json'
if IN_COLAB and not KAGGLE_JSON.exists():
    from google.colab import files
    print('Upload your kaggle.json'); files.upload()
    KAGGLE_JSON.parent.mkdir(exist_ok=True); shutil.move('kaggle.json', KAGGLE_JSON); os.chmod(KAGGLE_JSON, 0o600)
assert KAGGLE_JSON.exists() or os.environ.get('KAGGLE_KEY'), 'Kaggle token not found. Put kaggle.json in ' + str(KAGGLE_JSON.parent)
if not any((WORK / 'data').glob('*')):
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files('emmarex/plantdisease', path=str(WORK / 'data'), unzip=True, quiet=False)
print('data folder ready:', WORK / 'data')

In [ ]:
def find_root(base):
    for r, d, f in os.walk(base):
        if len(d) == 15 and all(any(x.lower().endswith(('.jpg','.jpeg','.png')) for x in os.listdir(os.path.join(r, c))) for c in d):
            return r
    raise RuntimeError('Could not find the folder with 15 class sub-folders')

ROOT = find_root(str(WORK / 'data'))
CLASSES = sorted(os.listdir(ROOT))
assert len(CLASSES) == 15, CLASSES
print(ROOT); print(*CLASSES, sep='\n')

def leaf_key(fname):
    # Group key: images of the same leaf (and its augmented copies) must stay on one side of the split.
    stem = Path(fname).stem
    base = stem.split('___')[0] if '___' in stem else stem
    return re.sub(r'[_ ]?(flip\w*|rot\w*|\d+deg)$', '', base, flags=re.I)

rows = []
for ci, c in enumerate(CLASSES):
    for f in sorted(os.listdir(os.path.join(ROOT, c))):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            rows.append((os.path.join(ROOT, c, f), ci, c + '/' + leaf_key(f)))
df = pd.DataFrame(rows, columns=['path', 'label', 'group'])
print(len(df), 'images |', df.group.nunique(), 'leaf groups')
if QUICK:
    df = df.groupby('label', group_keys=False).apply(lambda g: g.sample(min(len(g), 80), random_state=SEED)).reset_index(drop=True)
df.label.value_counts().sort_index().rename(index=dict(enumerate(CLASSES))).plot.barh(figsize=(7, 5), title='Images per class');

## 2. Leaf-level split and leak check
A random split puts different photos of the same leaf in both train and test, which inflates accuracy. We split by leaf group (stratified by class), then run a **near-duplicate check** (average-hash) between test and train images as an extra safety net.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = [te for _, te in sgkf.split(df, df.label, df.group)]
df['split'] = 'train'
df.loc[folds[0], 'split'] = 'test'; df.loc[folds[1], 'split'] = 'val'
g = {s: set(df[df.split == s].group) for s in ['train', 'val', 'test']}
assert not (g['train'] & g['test']) and not (g['train'] & g['val']) and not (g['val'] & g['test']), 'group leak!'
print(df.split.value_counts()); print('No leaf group appears in more than one split.')

In [ ]:
def ahash(path):
    im = Image.open(path); im.draft('L', (64, 64))
    a = np.asarray(im.convert('L').resize((8, 8), Image.BILINEAR), dtype=np.float32)
    bits = (a > a.mean()).flatten().astype(np.uint8)
    return np.packbits(bits).view(np.uint64)[0]

_POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def hamming_min(test_h, train_h, chunk=64):
    out = np.zeros(len(test_h), dtype=np.int32)
    for i in range(0, len(test_h), chunk):
        x = np.bitwise_xor(test_h[i:i+chunk, None], train_h[None, :])
        out[i:i+chunk] = _POP[x.view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(2).min(1)
    return out

t0 = time.time()
h = np.array([ahash(p) for p in df.path], dtype=np.uint64)
d = hamming_min(h[(df.split == 'test').values], h[(df.split == 'train').values])
print(f'hash time {time.time()-t0:.0f}s')
print(f'Test images with a near-duplicate (Hamming <= 4) in train: {(d <= 4).mean()*100:.2f}%')
print(f'Exact duplicates (distance 0): {(d == 0).mean()*100:.2f}%')
LEAK_PCT = float((d <= 4).mean() * 100)

## 3. Augmentation (incl. background replacement)
PlantVillage photos have plain backgrounds. `BGReplace` finds the leaf by colour saturation and pastes it onto random soil/grass/grey textures, so the network cannot rely on the background.

In [ ]:
class BGReplace:
    def __init__(self, p=0.5): self.p = p
    def _bg(self, h, w):
        kind = random.choice(['soil', 'grass', 'gray', 'noise'])
        base = np.array({'soil': [.35, .25, .17], 'grass': [.25, .45, .20], 'gray': [.6, .6, .6], 'noise': [.5, .5, .5]}[kind]) + np.random.uniform(-.1, .1, 3)
        bg = np.ones((h, w, 3), np.float32) * base
        yy, xx = np.ogrid[:h, :w]
        for _ in range(random.randint(8, 20)):
            cy, cx, r = random.randint(0, h), random.randint(0, w), random.randint(10, 60)
            bg[(yy - cy) ** 2 + (xx - cx) ** 2 < r * r] = np.clip(base + np.random.uniform(-.2, .2, 3), 0, 1)
        im = Image.fromarray((bg * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(random.uniform(2, 8)))
        return np.asarray(im).astype(np.float32) / 255
    def __call__(self, img):
        if random.random() > self.p: return img
        a = np.asarray(img.convert('RGB')).astype(np.float32) / 255
        mx, mn = a.max(2), a.min(2)
        sat = (mx - mn) / (mx + 1e-6)
        mask = ((sat > .18) & (mx > .12)).astype(np.uint8) * 255
        m = np.asarray(Image.fromarray(mask).filter(ImageFilter.GaussianBlur(2))).astype(np.float32)[..., None] / 255
        out = a * m + self._bg(*a.shape[:2]) * (1 - m)
        return Image.fromarray((out * 255).astype(np.uint8))

In [ ]:
import torchvision.transforms as T
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CFG['img']
tf_train = T.Compose([BGReplace(0.5), T.RandomResizedCrop(S, scale=(0.5, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.2),
                      T.RandomRotation(30), T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.3),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
tf_eval = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])   # same as the web app

class DS(Dataset):
    def __init__(self, frame, tf): self.p, self.y, self.tf = frame.path.tolist(), frame.label.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]

def loader(frame, tf, shuffle):
    return DataLoader(DS(frame, tf), batch_size=CFG['bs'], shuffle=shuffle, num_workers=CFG['workers'], pin_memory=True, drop_last=shuffle)

dtr, dva, dte = (df[df.split == s] for s in ['train', 'val', 'test'])
L_tr, L_va, L_te = loader(dtr, tf_train, True), loader(dva, tf_eval, False), loader(dte, tf_eval, False)

# preview augmentation
fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for i, a in enumerate(ax.flat):
    x = tf_train(Image.open(dtr.path.iloc[i * 37]).convert('RGB')) * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
    a.imshow(x.permute(1, 2, 0).clamp(0, 1)); a.axis('off')
plt.suptitle('Training augmentations (background replacement, crop, rotation, colour, blur)'); plt.show()

## 4. Models and training
EfficientNet-B0 (ImageNet pretrained). Phase 1 trains only the new classifier head; phase 2 fine-tunes everything with a smaller learning rate.

In [ ]:
import timm
class SimpleCNN(nn.Module):
    def __init__(self, n=15):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), blk(128, 256), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(256, n))
    def forward(self, x): return self.head(self.f(x))

def make_model(name):
    if name == 'simple_cnn': return SimpleCNN(15)
    return timm.create_model(name, pretrained=True, num_classes=15)

def head_params(m): return list((m.head if isinstance(m, SimpleCNN) else m.get_classifier()).parameters())
def freeze_backbone(m, freeze):
    for p in m.parameters(): p.requires_grad = not freeze
    for p in head_params(m): p.requires_grad = True

@torch.no_grad()
def predict(model, dl):
    model.eval().to(dev); lg, ys = [], []
    for x, y in dl:
        with torch.autocast(device_type=dev.type, enabled=dev.type == 'cuda'):
            lg.append(model(x.to(dev, non_blocking=True)).float().cpu())
        ys.append(y)
    return torch.cat(lg), torch.cat(ys)

def fit(model, epochs, lr, tag, train_loader=None, keep_best=True):
    train_loader = train_loader or L_tr
    model.to(dev)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_loader))
    scaler = torch.cuda.amp.GradScaler(enabled=dev.type == 'cuda')
    crit = nn.CrossEntropyLoss(label_smoothing=0.1); hist = []; best = (0, None)
    for ep in range(epochs):
        model.train(); t0 = time.time(); tl = 0; n = 0
        for x, y in train_loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.type, enabled=dev.type == 'cuda'):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y); n += len(y)
        lg, ys = predict(model, L_va); va = (lg.argmax(1) == ys).float().mean().item()
        hist.append(dict(epoch=len(hist) + 1, loss=tl / n, val_acc=va))
        if va > best[0]: best = (va, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        print(f'[{tag}] epoch {ep+1}/{epochs}  loss {tl/n:.4f}  val_acc {va:.4f}  ({time.time()-t0:.0f}s)')
    if keep_best: model.load_state_dict(best[1])
    return hist

if QUICK: CFG.update(epochs_head=1, epochs_ft=1, base_epochs_cnn=1, base_epochs_r50_head=1, base_epochs_r50_ft=1)

In [ ]:
seed_all()
effb0 = make_model('efficientnet_b0')
freeze_backbone(effb0, True);  h1 = fit(effb0, CFG['epochs_head'], CFG['lr_head'], 'effb0 head')
freeze_backbone(effb0, False); h2 = fit(effb0, CFG['epochs_ft'], CFG['lr_ft'], 'effb0 finetune')
torch.save(effb0.state_dict(), OUT / 'effb0_lab.pt')
hist = pd.DataFrame(h1 + h2); hist['epoch'] = range(1, len(hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
hist.plot(x='epoch', y='loss', ax=ax[0], title='Train loss'); hist.plot(x='epoch', y='val_acc', ax=ax[1], title='Validation accuracy'); plt.show()

In [ ]:
models = {'EfficientNet-B0 (ours)': effb0}
if RUN_BASELINES:
    seed_all(); cnn = make_model('simple_cnn'); fit(cnn, CFG['base_epochs_cnn'], 1e-3, 'simple CNN'); models['Simple CNN (from scratch)'] = cnn
    seed_all(); r50 = make_model('resnet50')
    freeze_backbone(r50, True);  fit(r50, CFG['base_epochs_r50_head'], CFG['lr_head'], 'resnet50 head')
    freeze_backbone(r50, False); fit(r50, CFG['base_epochs_r50_ft'], 1e-4, 'resnet50 finetune'); models['ResNet50 (transfer)'] = r50

## 5. Evaluation on the held-out lab test set

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
def metrics(lg, ys):
    p = lg.argmax(1).numpy(); y = ys.numpy()
    top3 = (lg.topk(3, 1).indices == ys[:, None]).any(1).float().mean().item()
    return dict(accuracy=accuracy_score(y, p), macro_f1=f1_score(y, p, average='macro'), top3=top3)

rows = []
for name, m in models.items():
    lg, ys = predict(m, L_te); rows.append(dict(model=name, params_M=sum(p.numel() for p in m.parameters()) / 1e6, **metrics(lg, ys)))
cmp = pd.DataFrame(rows).round(4); display(cmp)

LG_TE, Y_TE = predict(effb0, L_te)
print(classification_report(Y_TE, LG_TE.argmax(1), target_names=CLASSES, digits=3))
cm = confusion_matrix(Y_TE, LG_TE.argmax(1), normalize='true')
plt.figure(figsize=(9, 8)); plt.imshow(cm, cmap='Greens'); plt.colorbar(); plt.xticks(range(15), CLASSES, rotation=90, fontsize=7); plt.yticks(range(15), CLASSES, fontsize=7)
plt.title('Confusion matrix (row-normalised), lab test set'); plt.tight_layout(); plt.show()

## 6. Calibration
A model that says "94%" should be right about 94% of the time. We measure **ECE** (expected calibration error) and fix over-confidence with **temperature scaling** fitted on the validation set only.

In [ ]:
def ece(probs, labels, bins=15):
    conf, pred = probs.max(1); acc = (pred == labels).float(); e = 0.0
    edges = torch.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean().item() * abs(acc[m].mean().item() - conf[m].mean().item())
    return e

def fit_temperature(logits, labels):
    t = torch.zeros(1, requires_grad=True)   # T = exp(t) > 0
    opt = torch.optim.LBFGS([t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad(); l = F.cross_entropy(logits / t.exp(), labels); l.backward(); return l
    opt.step(closure); return t.exp().item()

LG_VA, Y_VA = predict(effb0, L_va)
TEMP = fit_temperature(LG_VA, Y_VA)
ece_b, ece_a = ece(F.softmax(LG_TE, 1), Y_TE), ece(F.softmax(LG_TE / TEMP, 1), Y_TE)
print(f'Temperature T = {TEMP:.3f}   ECE before {ece_b:.4f}  ->  after {ece_a:.4f}')

def reliability(logits, labels, T, ax, title, bins=10):
    p = F.softmax(logits / T, 1); conf, pred = p.max(1); acc = (pred == labels).float()
    xs, ys = [], []
    edges = torch.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 5: xs.append(conf[m].mean().item()); ys.append(acc[m].mean().item())
    ax.plot([0, 1], [0, 1], '--', c='gray'); ax.plot(xs, ys, 'o-', c='green'); ax.set_title(title); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
fig, ax = plt.subplots(1, 2, figsize=(9, 4)); reliability(LG_TE, Y_TE, 1.0, ax[0], 'Before'); reliability(LG_TE, Y_TE, TEMP, ax[1], 'After temperature scaling'); plt.show()

## 7. Field test (real photos): PlantDoc
Lab accuracy hides the real problem: photos taken in fields. PlantDoc has real-world images for some of our classes. We **never train on its test split**. First we measure the gap, then we fine-tune on the PlantDoc *train* split and measure again.

In [ ]:
PD = WORK / 'plantdoc'
if not PD.exists():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/pratikkayal/PlantDoc-Dataset', str(PD)], check=True)
assert (PD / 'train').exists() and (PD / 'test').exists(), os.listdir(PD)   # folder layout changed? adjust PD paths
RULES = [  # (required words in the PlantDoc folder name, our class)
 (['tomato', 'early'], 'Tomato_Early_blight'), (['tomato', 'late'], 'Tomato_Late_blight'), (['tomato', 'mold'], 'Tomato_Leaf_Mold'),
 (['tomato', 'septoria'], 'Tomato_Septoria_leaf_spot'), (['tomato', 'yellow'], 'Tomato__Tomato_YellowLeaf__Curl_Virus'),
 (['tomato', 'bacterial'], 'Tomato_Bacterial_spot'), (['tomato', 'mosaic'], 'Tomato__Tomato_mosaic_virus'),
 (['tomato', 'spider'], 'Tomato_Spider_mites_Two_spotted_spider_mite'),
 (['potato', 'early'], 'Potato___Early_blight'), (['potato', 'late'], 'Potato___Late_blight'),
 (['pepper', 'spot'], 'Pepper__bell___Bacterial_spot')]
HEALTHY = {'tomato leaf': 'Tomato_healthy', 'potato leaf': 'Potato___healthy', 'bell pepper leaf': 'Pepper__bell___healthy', 'bell_pepper leaf': 'Pepper__bell___healthy'}
def map_plantdoc(folder):
    n = folder.lower().replace('_', ' ').strip()
    if n in HEALTHY or folder.lower() in HEALTHY: return HEALTHY.get(n, HEALTHY.get(folder.lower()))
    for words, cls in RULES:
        if all(w in n for w in words): return cls
    return None

def pd_frame(split):
    rows, skipped = [], []
    for d in sorted((PD / split).iterdir()):
        if not d.is_dir(): continue
        cls = map_plantdoc(d.name)
        if cls is None: skipped.append(d.name); continue
        for f in d.iterdir():
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'): rows.append((str(f), CLASSES.index(cls), d.name))
    return pd.DataFrame(rows, columns=['path', 'label', 'pd_class']), skipped

fd_tr, sk1 = pd_frame('train'); fd_te, sk2 = pd_frame('test')
fd_val = fd_tr.sample(frac=0.2, random_state=SEED); fd_tr = fd_tr.drop(fd_val.index)   # field-val: used for model/threshold selection, never the test split
def _n(x): return re.sub(r'[^a-z0-9]', '', x.lower())
_LK = {_n(c): i for i, c in enumerate(CLASSES)}
def own_frame(folder):
    rows = []
    for d in sorted(Path(folder).iterdir()):
        if not d.is_dir(): continue
        if _n(d.name) not in _LK: print('skipping unknown folder:', d.name); continue
        rows += [(str(f), _LK[_n(d.name)], 'own/' + d.name) for f in d.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    return pd.DataFrame(rows, columns=['path', 'label', 'pd_class']).sample(frac=1, random_state=SEED)
if MY_PHOTOS_DIR and Path(MY_PHOTOS_DIR).exists():
    mine = own_frame(MY_PHOTOS_DIR); k = int(len(mine) * 0.8)
    fd_tr = pd.concat([fd_tr, mine.iloc[:k]]); fd_val = pd.concat([fd_val, mine.iloc[k:]]); print('own training photos added:', len(mine))
if MY_TEST_DIR and Path(MY_TEST_DIR).exists():
    mt = own_frame(MY_TEST_DIR); fd_te = pd.concat([fd_te, mt]); print('own TEST photos added:', len(mt))
print('PlantDoc images mapped: train', len(fd_tr), '| field-val', len(fd_val), '| field-test', len(fd_te)); print('Folders NOT mapped (ignored):', sk2)
display(fd_te.groupby('pd_class').size().to_frame('n_test'))

In [ ]:
L_fte = loader(fd_te, tf_eval, False)
def field_report(model, tag):
    lg, ys = predict(model, L_fte); m = metrics(lg, ys); print(f'{tag}: field accuracy {m["accuracy"]:.3f} | macro-F1 {m["macro_f1"]:.3f} | top-3 {m["top3"]:.3f}'); return lg, ys, m
lab_m = metrics(LG_TE, Y_TE)
LG_F0, Y_F, field0 = field_report(effb0, 'BEFORE field fine-tuning')
print(f'LAB accuracy {lab_m["accuracy"]:.3f}  ->  FIELD accuracy {field0["accuracy"]:.3f}   (gap {100*(lab_m["accuracy"]-field0["accuracy"]):.1f} points)')

### Narrowing the gap: fine-tune on PlantDoc train (+ a PlantVillage replay sample so the model does not forget the lab classes)

In [ ]:
from torch.utils.data import ConcatDataset
replay = dtr.groupby('label', group_keys=False).apply(lambda g: g.sample(min(len(g), 120), random_state=SEED))
mix = ConcatDataset([DS(fd_tr, tf_train)] * 3 + [DS(replay, tf_train)])
L_mix = DataLoader(mix, batch_size=CFG['bs'], shuffle=True, num_workers=CFG['workers'], pin_memory=True, drop_last=True)
seed_all()
effb0_field = make_model('efficientnet_b0'); effb0_field.load_state_dict(torch.load(OUT / 'effb0_lab.pt'))
freeze_backbone(effb0_field, False)
fit(effb0_field, 1 if QUICK else 4, 1e-4, 'field fine-tune', train_loader=L_mix, keep_best=False)
torch.save(effb0_field.state_dict(), OUT / 'effb0_field.pt')
LG_F1, _, field1 = field_report(effb0_field, 'AFTER field fine-tuning')
LG_TE2, _ = predict(effb0_field, L_te); lab2 = metrics(LG_TE2, Y_TE)
gap = pd.DataFrame([dict(model='Lab-trained', lab_acc=lab_m['accuracy'], field_acc=field0['accuracy']),
                    dict(model='+ field fine-tune', lab_acc=lab2['accuracy'], field_acc=field1['accuracy'])]).round(4)
display(gap); gap.plot.bar(x='model', rot=0, figsize=(6, 3.5), title='Lab vs field accuracy'); plt.show()

### 7b. (Optional) Compare with a community pretrained model
Set `PRETRAINED_DIR` in the config cell (folder made by `ml/download_pretrained.py`). Both models are scored on the same lab-test and field-test images. Caution: a community model may have seen your lab-test images during its own training (no leaf-level split), so its lab number can be inflated; the field-test comparison is the fairer one.

In [ ]:
def eval_onnx_dir(dirpath, frame):
    import onnxruntime as ort
    d = Path(dirpath); cfg = json.load(open(d / 'model_config.json', encoding='utf-8'))
    sess = ort.InferenceSession(str(d / cfg['onnx']), providers=['CPUExecutionProvider']); inp = sess.get_inputs()[0]
    size = int(inp.shape[2]) if isinstance(inp.shape[2], int) else cfg.get('input_size', 224)
    nrm = lambda t: re.sub(r'[^a-z0-9]', '', t.lower()); lk = {nrm(c): i for i, c in enumerate(CLASSES)}
    lk['tomatoyellowleafcurlvirus'] = CLASSES.index('Tomato__Tomato_YellowLeaf__Curl_Virus'); lk['tomatomosaicvirus'] = CLASSES.index('Tomato__Tomato_mosaic_virus')
    keep = [(i, lk[nrm(l)]) for i, l in enumerate(cfg['labels']) if nrm(l) in lk]; ki, kc = [i for i, _ in keep], [c for _, c in keep]
    tfm = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(cfg.get('mean', MEAN), cfg.get('std', STD))]); preds = []
    for i in range(0, len(frame), 32):
        x = torch.stack([tfm(Image.open(p).convert('RGB')) for p in frame.path.iloc[i:i + 32]]).numpy()
        preds += [kc[j] for j in sess.run(None, {inp.name: x})[0][:, ki].argmax(1)]
    return accuracy_score(frame.label.values, np.array(preds))

if PRETRAINED_DIR and Path(PRETRAINED_DIR).exists():
    cmp2 = pd.DataFrame([dict(model='Community pretrained (as downloaded)', lab_acc=eval_onnx_dir(PRETRAINED_DIR, dte), field_acc=eval_onnx_dir(PRETRAINED_DIR, fd_te))] + gap.to_dict('records')).round(4)
    display(cmp2); cmp2.to_csv(OUT / 'community_vs_ours.csv', index=False)
else:
    print('PRETRAINED_DIR not set: skipping the community-model comparison.')

## 8. Model choice, calibration and confidence threshold
Everything here is decided on the **field-val** split (carved out of PlantDoc *train*), then reported once on the untouched **field-test** split. Above the threshold the app answers; below it, it asks for a retake. A higher threshold gives fewer but more reliable answers.

In [ ]:
L_fva = loader(fd_val, tf_eval, False)
def facc(m): lg, ys = predict(m, L_fva); return (lg.argmax(1) == ys).float().mean().item()
FINAL = effb0_field if facc(effb0_field) >= facc(effb0) else effb0
print('Model used for export:', 'field fine-tuned' if FINAL is effb0_field else 'lab-trained', '(chosen on field-val)')
lg_v, y_v = predict(FINAL, L_fva); TEMP = fit_temperature(lg_v, y_v); print('Final temperature (fitted on field-val):', round(TEMP, 3))
lgF, yF = predict(FINAL, L_fte)
print(f'Field-test ECE before {ece(F.softmax(lgF, 1), yF):.4f} -> after {ece(F.softmax(lgF / TEMP, 1), yF):.4f}')

def rc_table(lg, ys, T):
    conf, pred = F.softmax(lg / T, 1).max(1); ok = (pred == ys).float(); rows = []
    for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        m = conf >= t; rows.append(dict(threshold=t, coverage=m.float().mean().item(), accuracy_when_answering=ok[m].mean().item() if m.any() else float('nan')))
    return pd.DataFrame(rows).round(3)
rc_val, rc = rc_table(lg_v, y_v, TEMP), rc_table(lgF, yF, TEMP)
TARGET = 0.85
good = rc_val[rc_val.accuracy_when_answering >= TARGET]
THR = float(good.threshold.min()) if len(good) else 0.6
print(f'Chosen threshold = {THR} (lowest value where field-val accuracy on answered photos >= {TARGET})')
print('field-val'); display(rc_val); print('field-test (reported)'); display(rc)
plt.plot(rc.coverage, rc.accuracy_when_answering, 'o-', label='field-test'); [plt.annotate(f'{t:.1f}', (c, a)) for t, c, a in zip(rc.threshold, rc.coverage, rc.accuracy_when_answering)]
plt.xlabel('coverage (share of photos answered)'); plt.ylabel('accuracy on answered photos'); plt.title('Risk-coverage curve'); plt.grid(alpha=.3); plt.legend(); plt.show()

## 9. Grad-CAM: what is the model looking at?

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
FINAL.to(dev).eval()
imgs = fd_te.sample(4, random_state=1).path.tolist() + dte.sample(4, random_state=1).path.tolist()
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
with GradCAM(model=FINAL, target_layers=[FINAL.blocks[-1]]) as cam:
    for a, p in zip(ax.flat, imgs):
        im = Image.open(p).convert('RGB').resize((S, S)); x = tf_eval(im).unsqueeze(0).to(dev)
        g = cam(input_tensor=x)[0]; a.imshow(show_cam_on_image(np.asarray(im) / 255., g, use_rgb=True)); a.axis('off')
        a.set_title(CLASSES[FINAL(x).argmax().item()].replace('_', ' ')[:28], fontsize=8)
plt.suptitle('Grad-CAM (top row: field photos, bottom row: lab photos)'); plt.show()

## 10. Export to ONNX and verify

In [ ]:
import onnx, onnxruntime as ort
try:
    timm.layers.set_exportable(True)
except Exception as e: print('set_exportable skipped:', e)
FINAL.eval().cpu()
dummy = torch.randn(1, 3, S, S)
ONNX_PATH = OUT / 'cropguard_effb0.onnx'
_kw = dict(input_names=['input'], output_names=['logits'], dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
try:
    torch.onnx.export(FINAL, dummy, str(ONNX_PATH), dynamo=False, **_kw)      # classic exporter (newer PyTorch defaults to a different one)
except Exception:
    torch.onnx.export(FINAL, dummy, str(ONNX_PATH), **_kw)
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
xb = torch.randn(8, 3, S, S)
with torch.no_grad(): ref = FINAL(xb).numpy()
out = sess.run(None, {'input': xb.numpy()})[0]
print('max |torch - onnx| =', float(np.abs(ref - out).max()))
t0 = time.time(); [sess.run(None, {'input': xb[:1].numpy()}) for _ in range(20)]
print(f'CPU latency per image: {(time.time()-t0)/20*1000:.0f} ms | file size {ONNX_PATH.stat().st_size/1e6:.1f} MB')

In [ ]:
DISPLAY = {'Pepper__bell___Bacterial_spot': ('Pepper', 'Bacterial Spot'), 'Pepper__bell___healthy': ('Pepper', 'Healthy'),
 'Potato___Early_blight': ('Potato', 'Early Blight'), 'Potato___Late_blight': ('Potato', 'Late Blight'), 'Potato___healthy': ('Potato', 'Healthy'),
 'Tomato_Bacterial_spot': ('Tomato', 'Bacterial Spot'), 'Tomato_Early_blight': ('Tomato', 'Early Blight'), 'Tomato_Late_blight': ('Tomato', 'Late Blight'),
 'Tomato_Leaf_Mold': ('Tomato', 'Leaf Mold'), 'Tomato_Septoria_leaf_spot': ('Tomato', 'Septoria Leaf Spot'),
 'Tomato_Spider_mites_Two_spotted_spider_mite': ('Tomato', 'Spider Mites'), 'Tomato__Target_Spot': ('Tomato', 'Target Spot'),
 'Tomato__Tomato_YellowLeaf__Curl_Virus': ('Tomato', 'Yellow Leaf Curl Virus'), 'Tomato__Tomato_mosaic_virus': ('Tomato', 'Mosaic Virus'), 'Tomato_healthy': ('Tomato', 'Healthy')}
def pretty(c):
    crop, dis = DISPLAY.get(c, (c.split('_')[0], c.replace('_', ' ').strip()))
    return dict(crop=crop, disease=dis)
json.dump(dict(classes=CLASSES, display={c: pretty(c) for c in CLASSES}, input_size=S, mean=MEAN, std=STD), open(OUT / 'class_names.json', 'w'), indent=2)
json.dump(dict(temperature=TEMP, confidence_threshold=THR), open(OUT / 'calibration.json', 'w'), indent=2)
json.dump(dict(lab_test=cmp.to_dict('records'), lab_vs_field=gap.to_dict('records'), ece_before=ece_b, ece_after=ece_a,
               near_duplicate_pct=LEAK_PCT, risk_coverage_field_test=rc.to_dict('records'), risk_coverage_field_val=rc_val.to_dict('records'), threshold=THR,
               split_sizes=df.split.value_counts().to_dict(), seed=SEED), open(OUT / 'metrics.json', 'w'), indent=2)
cmp.to_csv(OUT / 'model_comparison.csv', index=False); gap.to_csv(OUT / 'lab_vs_field.csv', index=False)
zip_path = shutil.make_archive(str(WORK / 'cropguard_artifacts'), 'zip', OUT)
print(sorted(os.listdir(OUT)))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print('Zip saved at', zip_path)

## Install the new model into the CropGuard project (local runs)
On Colab this is skipped: download the zip instead and follow the guide (unzip into `ml/models/cropguard/`, then `python ml/make_model_config.py`).

In [ ]:
if not IN_COLAB:
    proj = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend' / 'app').exists() and (p / 'ml').exists()), None)
    if proj is None:
        print('CropGuard project folder not found from', Path.cwd(), '- copy', OUT, 'into <project>/ml/models/cropguard manually.')
    else:
        dst = proj / 'ml' / 'models' / 'cropguard'; dst.mkdir(parents=True, exist_ok=True)
        for f in list(OUT.glob('cropguard_effb0.onnx*')) + [OUT / 'class_names.json', OUT / 'calibration.json', OUT / 'metrics.json']:
            shutil.copy(f, dst / f.name)
        subprocess.run([sys.executable, str(proj / 'ml' / 'make_model_config.py')], check=True)
        print('Installed into', dst, '-> now restart the server.')

## What to copy into your report
- **Table 1** (`model_comparison.csv`): Simple CNN vs ResNet50 vs EfficientNet-B0 on the held-out lab test set (accuracy, macro-F1, top-3).
- **Table 2** (`lab_vs_field.csv`): lab accuracy vs field accuracy before and after field fine-tuning. Report the gap honestly.
- **Figures:** confusion matrix, reliability diagram, risk-coverage curve, Grad-CAM grid, augmentation preview.
- **Method notes:** leaf-level split plus the near-duplicate check (`near_duplicate_pct` in `metrics.json`), temperature scaling, and the threshold chosen on field-val and reported on field-test.
- **Limits to state:** PlantVillage is lab-style and has no pests except spider mites; PlantDoc covers only some classes; severity is an image-based estimate, not a trained model.

Next: give `cropguard_effb0.onnx`, `class_names.json` and `calibration.json` to the backend. Preprocessing in the app must match `tf_eval`: resize to 224x224, scale to 0-1, normalise with the mean/std in `class_names.json`, then apply `softmax(logits / temperature)` and use `confidence_threshold` from `calibration.json` for the retake gate (the web app currently shows a 60% placeholder).